# Week 3 · Class 1 — Embeddings: Meaning as Numbers

Turn text into vectors, measure meaning with cosine similarity, and build a working semantic
search engine over your own Week 2 chunks — using nothing but `sentence-transformers` and NumPy.

## Before you start

This notebook is standalone and runs top to bottom with **no API key** — the embedding model runs
locally in Colab. If you have your Week 2 `data/chunks.json`, upload it when you reach Section 6;
if not, a built-in fallback corpus keeps every cell runnable.

By the end you will have:

1. Computed cosine similarity by hand and with NumPy
2. Seen meaning cluster in a similarity matrix
3. Built `search(query, k)` — the heart of every RAG system
4. Plotted your corpus in 2-D with PCA
5. Found the places where embeddings *fail* (this matters as much as where they work)

## Section 1 — Install packages

`sentence-transformers` brings the embedding model; the rest is for math and plots.

In [ ]:
!pip install -q sentence-transformers scikit-learn matplotlib pandas

## Section 2 — Load the embedding model

`all-MiniLM-L6-v2` is the workhorse of open-source embeddings: ~80 MB, fast on CPU, and it maps
**any text — a word, a sentence, a whole chunk — to exactly 384 numbers**.

The first run downloads the model (once per Colab session).

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

vec = model.encode("I love pizza")
print(type(vec), vec.shape)   # numpy array, (384,)
print("first 5 numbers:", vec[:5])

That array **is the embedding**. Two properties make it useful:

- **Fixed length** — every input becomes 384 numbers, so any two texts are comparable
- **Similar meaning → nearby vectors** — the model was trained on ~1B sentence pairs to make this true

## Section 3 — Cosine similarity from scratch

How close are two vectors? Three common metrics:

| Metric | Formula | Intuition |
|--------|---------|-----------|
| Dot product | $\sum a_i b_i$ | alignment × magnitude |
| **Cosine similarity** | $\dfrac{a \cdot b}{\|a\|\,\|b\|}$ | **angle only** — 1 = same direction, 0 = unrelated |
| Euclidean (L2) | $\|a-b\|$ | straight-line distance |

Cosine is the default for text. Let's build it ourselves before trusting a library.

In [ ]:
import numpy as np

def cosine(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# 2-D warm-up — small enough to check on paper
a = [3, 4]    # imagine: "pizza"
b = [4, 3]    # imagine: "pasta"
c = [-4, 3]   # imagine: "tax law"

print("cos(a, b) =", round(cosine(a, b), 3))   # 0.96  — very similar
print("cos(a, c) =", round(cosine(a, c), 3))   # 0.0   — unrelated
print("cos(a, a) =", round(cosine(a, a), 3))   # 1.0   — identical direction

### 🔹 Mini-exercise (5 min, pen and paper first!)

Compute the cosine similarity of `u = [1, 2, 2]` and `v = [2, 2, 1]` **by hand**:

1. Dot product: $1{\cdot}2 + 2{\cdot}2 + 2{\cdot}1 = ?$
2. Norms: $\|u\| = \sqrt{1+4+4} = ?$ and $\|v\| = ?$
3. Divide.

Write your answer down, *then* run the check below.

In [ ]:
u = [1, 2, 2]
v = [2, 2, 1]

my_answer = ...   # TODO: type the number you computed by hand (3 decimals is fine)

check = cosine(u, v)
print("NumPy says:", round(check, 4))
if isinstance(my_answer, float) and abs(my_answer - check) < 0.01:
    print("✅ Your hand computation matches!")
else:
    print("✍️  Compare with your paper result — where did it diverge?")

## Section 4 — Similarity matrix: watch meaning cluster

Six sentences, three topics. Before running: **predict** which pairs will score high.

In [ ]:
import pandas as pd

sentences = [
    "I love pizza",
    "Pizza is my favorite food",
    "The World Cup final was thrilling",
    "Football is the most popular sport in Brazil",
    "Python is a programming language",
    "I write code in Python every day",
]

# normalize_embeddings=True → every vector has length 1 → dot product == cosine
embs = model.encode(sentences, normalize_embeddings=True)
sim = embs @ embs.T

labels = [s[:28] + "…" if len(s) > 28 else s for s in sentences]
df = pd.DataFrame(np.round(sim, 2), index=labels, columns=labels)
df.style.background_gradient(cmap="viridis", vmin=0, vmax=1)

Look for the **2×2 bright blocks** along the diagonal: food×food, sports×sports,
python×python all score ~0.5–0.8, while cross-topic pairs sit near 0.0–0.2.

The model never saw these sentences together — the geometry alone recovered the topics.

### 🔹 Mini-exercise (3 min)

Add two sentences of your own to the list — one that should join an existing cluster and one
about a brand-new topic. **Predict the scores before re-running the cell.** Were you right?

## Section 5 — Semantic search in 15 lines

Embed a corpus once, then answer queries by cosine. This function is the heart of every
RAG system you will ever build.

In [ ]:
corpus = [
    "Automobile repair pricing starts at 80 pounds per hour at our garage.",
    "The 2026 FIFA World Cup will be hosted by the USA, Canada, and Mexico.",
    "Python 3.13 introduced an experimental free-threaded mode without the GIL.",
    "Our refund policy allows returns within 30 days with a valid receipt.",
    "The goalkeeper made three incredible saves in the penalty shootout.",
    "To install packages, run pip install followed by the package name.",
    "Deep dish pizza originated in Chicago in the 1940s.",
    "Student visas require proof of enrollment and financial support documents.",
]

corpus_embs = model.encode(corpus, normalize_embeddings=True)   # (N, 384)

def search(query: str, k: int = 3, embs=corpus_embs, texts=corpus):
    q = model.encode(query, normalize_embeddings=True)          # (384,)
    scores = embs @ q                                           # (N,) cosines
    top = np.argsort(scores)[::-1][:k]
    return [(float(scores[i]), texts[i]) for i in top]

for score, text in search("how much does it cost to fix my car?"):
    print(f"{score:.3f}  {text}")

**Notice**: the query says *fix my car* — the top hit says *automobile repair*.
**Zero shared keywords.** This is the exact miss that keyword search cannot recover from.

### 🔹 Mini-exercise (5 min)

For each corpus entry, write a query that shares **no words** with it but still retrieves it
as the top-1 hit. Try at least three. Then find one query where semantic search returns
something *wrong* — and explain why in a comment.

In [ ]:
# Your no-keyword-overlap queries:
for q in [
    "who is playing in the next mundial?",      # → World Cup (works even in Spanish-ish!)
    # TODO: add yours here
]:
    print(f"Q: {q}")
    for score, text in search(q, k=1):
        print(f"   {score:.3f}  {text}")
    print()

## Section 6 — Your Week 2 data

Time to search **your own dataset**. In Week 2 Class 4 you saved `data/chunks.json`
(PDF chunks) and scraped JSON files.

**Colab:** run the next cell, then use the file-upload widget (or drag files into the Files
sidebar and adjust the path). **No Week 2 files handy?** The cell silently falls back to a
built-in mini-corpus so everything below still runs.

In [ ]:
import json, os

FALLBACK_CHUNKS = [
    {"text": t, "source": "fallback"} for t in corpus
]

chunks = None
for path in ["data/chunks.json", "chunks.json"]:
    if os.path.exists(path):
        raw = json.load(open(path))
        # Week 2 saved either a list of strings or a list of dicts — handle both
        chunks = [
            {"text": c, "source": "chunks.json"} if isinstance(c, str) else c
            for c in raw
        ]
        print(f"Loaded {len(chunks)} chunks from {path}")
        break

if chunks is None:
    chunks = FALLBACK_CHUNKS
    print(f"No Week 2 file found — using {len(chunks)} fallback chunks. "
          "Upload data/chunks.json to use your own data.")

In [ ]:
texts = [c["text"] for c in chunks]
chunk_embs = model.encode(texts, normalize_embeddings=True, show_progress_bar=True)
print("Embedded:", chunk_embs.shape)

# Ask something YOUR dataset can answer — edit these!
for q in [
    "what does the document say about pricing?",
    "tell me about the tournament groups",
]:
    print(f"\nQ: {q}")
    for score, text in search(q, k=2, embs=chunk_embs, texts=texts):
        print(f"   {score:.3f}  {text[:90]}")

**Calibrate your intuition:** run a few queries and watch the scores.
As a rough guide for MiniLM: **0.6+** = strong match, **0.35–0.6** = related,
**below ~0.3** = the corpus probably doesn't contain an answer. You'll use a threshold like
this in Class 3 to make your agent say "I don't know" honestly.

## Section 7 — Seeing 384-D in 2-D with PCA

We can't picture 384 dimensions, but PCA finds the 2 directions with the most variance and
projects everything onto them. Clusters you *measured* in Section 4 become clusters you can *see*.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

topic_sentences = {
    "food":   ["I love pizza", "Pasta with garlic butter", "Sushi is delicate",
               "Deep dish pizza from Chicago", "Fresh croissants for breakfast"],
    "sports": ["The World Cup final", "The goalkeeper saved a penalty",
               "Champions League anthem", "A last-minute goal won the match",
               "The striker was offside"],
    "python": ["Python list comprehensions", "pip install numpy",
               "Debugging a TypeError", "Async functions with await",
               "Virtual environments keep dependencies clean"],
}

all_texts, all_topics = [], []
for topic, sents in topic_sentences.items():
    all_texts += sents
    all_topics += [topic] * len(sents)

coords = PCA(n_components=2).fit_transform(model.encode(all_texts))

colors = {"food": "#34d399", "sports": "#22d3ee", "python": "#8b5cf6"}
plt.figure(figsize=(9, 6), facecolor="white")
for topic in topic_sentences:
    idx = [i for i, t in enumerate(all_topics) if t == topic]
    plt.scatter(coords[idx, 0], coords[idx, 1], c=colors[topic], s=80, label=topic)
for i, txt in enumerate(all_texts):
    plt.annotate(txt[:22], coords[i], fontsize=7, alpha=0.6)
plt.legend(); plt.title("384-D embeddings, squashed to 2-D with PCA")
plt.tight_layout(); plt.show()

Three islands. The model organized meaning into geometry — and PCA let us peek.

> For larger corpora, t-SNE or UMAP make prettier maps; PCA is enough for intuition.

## Section 8 — Where embeddings fail (limitations lab)

Knowing the failure modes is what separates "I used embeddings" from "I understand retrieval".

In [ ]:
pairs = [
    # (a, b, why we test it)
    ("The pizza was great",   "The pizza was not great",  "NEGATION"),
    ("Order number 48291",    "Order number 48292",       "EXACT IDs"),
    ("The meeting is at 3pm", "The meeting is at 4pm",    "NUMBERS"),
]

for a, b, label in pairs:
    ea, eb = model.encode([a, b], normalize_embeddings=True)
    print(f"{label:>10}:  cos = {float(ea @ eb):.3f}   ({a!r} vs {b!r})")

All three pairs score **very high** — yet each pair means something importantly
different. Lessons:

- **Negation** barely moves the vector → the LLM must re-read retrieved text (it does, in RAG)
- **Exact identifiers** are nearly identical in embedding space → you need keyword/hybrid search (Week 4)
- **Numbers** don't behave like quantities → you need metadata filters (Class 3: Qdrant payloads)

Retrieval finds *relevant* text. It does not certify *true* or *exact* — design around that.

## Section 9 — Homework

1. **Hand-cosine** — finish the Section 3 exercise if you haven't; be able to do a 3-D cosine on paper.
2. **Break the search engine** — find **3 queries** on your Week 2 data where the top-1 hit is wrong.
   For each, one sentence: *why* did the geometry betray you?
3. **Cross-lingual probe** — load `paraphrase-multilingual-MiniLM-L12-v2` and embed the same
   sentence in English and your own language. Report the cosine. (This model maps 50+ languages
   into one shared space.)
4. **Stretch** — PCA-plot your entire Week 2 chunk set, colored by source (blog / wiki / PDF).
   Do sources form clusters? Should they?

## Checklist before Class 2

- [ ] I can compute cosine similarity on paper for small vectors
- [ ] `search(query, k)` runs on my own Week 2 chunks
- [ ] I know three failure modes of embeddings and their fixes
- [ ] My `chunks.json` + scraped datasets are saved where I can find them — Class 2 indexes them with FAISS